## Sesion 5 - Pytest Avanzado: Fixtures, Parametrize y Mocks

**Bootcamp:** Python PY4  
**Modulo 4:** Big Data y MLOps  
**Taller de Calidad: Pruebas Unitarias y Documentación Profesional**

---

En la sesion pasada escribimos nuestras primeras pruebas: funciones que empiezan con `test_`, y que usan `assert` para verificar un resultado. Eso alcanza para empezar, pero a medida que un proyecto crece, escribir pruebas asi se vuelve repetitivo y limitado.  
Hoy vamos a ver tres herramientas de pytest que resuelven problemas muy concretos que aparecen apenas se escriben mas de un par de pruebas: **fixtures**, **parametrize** y **mocks**.

## Objetivos de la sesion

Al final de esta sesion vas a poder:

- Reconocer cuando una prueba tiene codigo de preparacion repetido, y resolverlo con un fixture.
- Usar `parametrize` para probar varios casos con una sola funcion de test, en vez de copiar y pegar.
- Explicar por que no se debe probar codigo que depende de un servicio externo real, y usar un mock para resolverlo.
- (Bonus) medir que porcentaje del codigo del proyecto esta efectivamente cubierto por pruebas.

## Parte 0: Retomando el proyecto

Montamos Drive, nos ubicamos en `taller_calidad`, e instalamos las dependencias del proyecto, ya que estamos en un runtime nuevo de Colab.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_NAME = "taller_calidad"
PROJECT_PATH = f"/content/drive/MyDrive/{PROJECT_NAME}"

os.chdir(PROJECT_PATH)

print("Trabajando en:", os.getcwd())

Trabajando en: /content/drive/MyDrive/taller_calidad


In [3]:
!pip install -r requirements.txt -q

Confirmemos que las pruebas de las sesiones anteriores siguen pasando, antes de agregar nada nuevo.

In [13]:
!python -m pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/taller_calidad
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 4 items                                                              

tests/test_marketing.py::test_calcular_roi_exitoso PASSED                [ 25%]
tests/test_marketing.py::test_calcular_roi_inversion_cero PASSED         [ 50%]
tests/test_procesamiento.py::test_calcular_metricas_con_datos PASSED     [ 75%]
tests/test_procesamiento.py::test_calcular_metricas_dataframe_vacio PASSED [100%]

============================== 4 passed in 1.88s ===============================


## Parte 1: Fixtures - evitando repetir la preparacion

### El problema

Abramos `tests/test_procesamiento.py` tal como quedo en la sesion 3.

In [14]:
!cat tests/test_procesamiento.py

"""Pruebas unitarias para src/procesamiento.py"""

import pandas as pd  # Importamos Pandas para crear DataFrames de prueba

from src.procesamiento import calcular_metricas_ventas  # Importamos la función que vamos a probar


def test_calcular_metricas_con_datos():
    # Creamos un DataFrame con datos de ventas para probar el caso normal
    df = pd.DataFrame({
        "producto": ["A", "B"],
        "cantidad": [2, 1],
        "precio_unitario": [10.0, 20.0],
    })

    resultado = calcular_metricas_ventas(df)  # Ejecutamos la función con los datos de prueba

    # Verificamos que las métricas calculadas sean las esperadas
    assert resultado["total_ventas"] == 40.0
    assert resultado["cantidad_transacciones"] == 2
    assert resultado["ticket_promedio"] == 20.0


def test_calcular_metricas_dataframe_vacio():
    # Creamos un DataFrame vacío para probar el caso sin datos
    df = pd.DataFrame(columns=["producto", "cantidad", "precio_unitario"])

    resultado = calcular_metricas_v

Fijate que las dos pruebas arman su propio `DataFrame` desde cero, con columnas parecidas. Si manana cambia el nombre de una columna en el proyecto real (por ejemplo, `precio_unitario` pasa a llamarse `precio`), hay que actualizarlo en cada prueba, una por una. Con dos pruebas no es un problema. Con veinte, si.

Esto es una señal comun en programacion: **cuando el mismo bloque de preparacion aparece en varios lugares, conviene escribirlo una sola vez y reutilizarlo.**

### La idea de un fixture

Pensalo como la mise en place en una cocina: antes de cocinar varias recetas, alguien pica la cebolla y la deja lista en un bowl. Cada receta que necesite cebolla picada la toma de ahi, en vez de que cada cocinero la pique por su cuenta. Un **fixture** en pytest es exactamente eso: una funcion que prepara algo (datos, una conexion, un objeto) una sola vez, y que cualquier prueba puede pedir con solo nombrarla.

### `conftest.py`: donde viven los fixtures compartidos

pytest tiene una convencion especial: si creamos un archivo llamado `conftest.py` dentro de `tests/`, cualquier fixture que definamos ahi queda disponible automaticamente para todas las pruebas de esa carpeta, sin necesidad de importarlo. Vamos a crear uno con un fixture que arma un dataset de ventas de ejemplo.

In [15]:
# Creamos el archivo con las fixtures compartidas entre las pruebas
%%writefile tests/conftest.py
"""Fixtures compartidos entre los archivos de prueba del proyecto."""  # Describimos el objetivo del módulo

import pandas as pd
import pytest


@pytest.fixture  # Indicamos que esta función será una fixture de pytest
def dataset_ventas_ejemplo():
    """Devuelve un DataFrame de ventas de ejemplo, listo para usar en varias pruebas."""  # Documentamos qué proporciona la fixture
    return pd.DataFrame({  # Creamos y devolvemos un DataFrame de ejemplo
        "producto": ["A", "B"],
        "cantidad": [2, 1],
        "precio_unitario": [10.0, 20.0],

    })

Overwriting tests/conftest.py


Ahora refactoricemos `tests/test_procesamiento.py` para que use este fixture en vez de armar el DataFrame a mano. Fijate en la primera prueba: el parametro `dataset_ventas_ejemplo` tiene el mismo nombre que el fixture que acabamos de crear. Eso no es casualidad, es la forma en que pytest sabe que dato inyectar: ve el nombre del parametro, busca un fixture con ese nombre (en este archivo o en `conftest.py`), lo ejecuta, y te entrega lo que devuelve. No hace falta importar nada.

In [16]:
%%writefile tests/test_procesamiento.py
"""Pruebas unitarias para src/procesamiento.py"""

import pandas as pd

from src.procesamiento import calcular_metricas_ventas


def test_calcular_metricas_con_datos(dataset_ventas_ejemplo):
    resultado = calcular_metricas_ventas(dataset_ventas_ejemplo)

    assert resultado["total_ventas"] == 40.0
    assert resultado["cantidad_transacciones"] == 2
    assert resultado["ticket_promedio"] == 20.0


def test_calcular_metricas_dataframe_vacio():
    df = pd.DataFrame(columns=["producto", "cantidad", "precio_unitario"])

    resultado = calcular_metricas_ventas(df)

    assert resultado["total_ventas"] == 0.0
    assert resultado["ticket_promedio"] == 0.0
    assert resultado["cantidad_transacciones"] == 0

Overwriting tests/test_procesamiento.py


Corramos las pruebas para confirmar que el fixture funciono correctamente.

In [17]:
!python -m pytest tests/test_procesamiento.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/taller_calidad
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 2 items                                                              

tests/test_procesamiento.py::test_calcular_metricas_con_datos PASSED     [ 50%]
tests/test_procesamiento.py::test_calcular_metricas_dataframe_vacio PASSED [100%]

============================== 2 passed in 0.07s ===============================


La segunda prueba (`test_calcular_metricas_dataframe_vacio`) no pidio el fixture, porque justamente necesita un DataFrame vacio, no el de ejemplo. Un fixture es opcional por prueba: cada funcion de test solo pide los que realmente necesita, nombrandolos como parametros.

## Parte 2: Parametrize - un test, varios casos

### El problema

Supongamos que queremos probar `calcular_iva` con varios montos distintos, no solo uno. La forma que ya conocemos seria escribir una funcion de test por cada caso:

```python
def test_calcular_iva_caso_100():
    assert calcular_iva(100) == 13.0

def test_calcular_iva_caso_200():
    assert calcular_iva(200) == 26.0

def test_calcular_iva_caso_50():
    assert calcular_iva(50, tasa=0.10) == 5.0
```

Funciona, pero son tres funciones casi identicas, y si mañana hay que cambiar algo de la logica de la prueba (por ejemplo, agregar un redondeo), hay que tocar las tres. Es el mismo problema de repeticion que vimos con los fixtures, aplicado ahora a los casos de prueba en vez de a la preparacion de datos.

### La idea de parametrize

`@pytest.mark.parametrize` es como un for loop para pruebas: en vez de escribir la receta de prueba varias veces con distintos ingredientes, se escribe **una sola vez**, y se le entrega una lista de combinaciones de valores para probar. pytest corre la misma funcion una vez por cada combinacion.

Veamos como se ve esto con `calcular_iva`, definido aqui mismo en el notebook a modo de demostracion.

In [18]:
import pytest

# Definimos la función que calcula el IVA
def calcular_iva(monto, tasa=0.13):
    if monto < 0:  # Verificamos que el monto no sea negativo
        raise ValueError("El monto no puede ser negativo")  # Generamos un error si el monto es inválido
    return round(monto * tasa, 2)  # Calculamos el IVA y redondeamos a dos decimales


# Indicamos que esta prueba se ejecutará con varios conjuntos de datos
@pytest.mark.parametrize(
    "monto, tasa, iva_esperado",  # Definimos los parámetros que recibirá la función de prueba
    [
        (100, 0.13, 13.0),  # Caso 1: IVA del 13% sobre 100
        (200, 0.13, 26.0),  # Caso 2: IVA del 13% sobre 200
        (50, 0.10, 5.0),  # Caso 3: IVA del 10% sobre 50
    ],
)

# Definimos la prueba con sus parámetros
def test_calcular_iva_varios_casos(monto, tasa, iva_esperado):
    assert calcular_iva(monto, tasa) == iva_esperado  # Comprobamos que el resultado coincida con el IVA esperado


# Simulamos lo que pytest haría automáticamente: ejecutar la función una vez por cada caso
for monto, tasa, iva_esperado in [(100, 0.13, 13.0), (200, 0.13, 26.0), (50, 0.10, 5.0)]:
    test_calcular_iva_varios_casos(monto, tasa, iva_esperado)  # Ejecutamos la prueba con los valores del caso actual

print("Los tres casos pasaron correctamente")  # Mostramos un mensaje cuando todos los casos pasan correctamente

Los tres casos pasaron correctamente


Si hubieramos corrido esto con pytest de verdad (en vez de simularlo con un for), cada caso aparece como una prueba separada en el resultado, algo como:

```text
test_calcular_iva_varios_casos[100-0.13-13.0] PASSED
test_calcular_iva_varios_casos[200-0.13-26.0] PASSED
test_calcular_iva_varios_casos[50-0.1-5.0] PASSED
```

Esto tiene una ventaja practica importante: si uno de los tres casos falla, el nombre te dice exactamente cual, sin tener que adivinar.

Ahora, de forma mas real. Vamos a reescribir `tests/test_marketing.py` para probar `calcular_roi` con varios pares de ingresos e inversion en una sola funcion, en vez de una funcion de test por caso.

In [19]:
# Creamos el archivo con las pruebas del módulo marketing
%%writefile tests/test_marketing.py
"""Pruebas unitarias para src/marketing.py"""  # Describimos el objetivo de este archivo

import pytest  # Importamos pytest para crear y ejecutar las pruebas

from src.marketing import calcular_roi  # Importamos la función que queremos probar


@pytest.mark.parametrize(  # Indicamos que esta prueba se ejecutará varias veces con diferentes valores
    "ingresos, inversion, roi_esperado",  # Definimos los nombres de los parámetros que recibirá la prueba
    [
        (1500, 1000, 0.5),  # Caso 1: ingresos de 1500, inversión de 1000 y ROI esperado de 0.5
        (1000, 1000, 0.0),  # Caso 2: ingresos iguales a la inversión y ROI esperado de 0.0
        (800, 1000, -0.2),  # Caso 3: ingresos menores que la inversión y ROI esperado de -0.2
    ],
)# fijate que la funcion viene directamente abajo del @pytest.mark.parametrize
def test_calcular_roi_varios_casos(ingresos, inversion, roi_esperado):  # Recibimos los valores de cada caso proporcionados por parametrize
    assert calcular_roi(ingresos, inversion) == roi_esperado  # Comprobamos que el ROI calculado coincida con el valor esperado


def test_calcular_roi_inversion_cero():  # Probamos el caso en que la inversión es cero
    with pytest.raises(ValueError):  # Indicamos que esperamos que se genere un ValueError
        calcular_roi(1500, 0)  # Intentamos calcular el ROI con una inversión inválida

Overwriting tests/test_marketing.py


Notemos que dejamos tres casos deliberadamente distintos: uno con ganancia, uno sin ganancia ni perdida, y uno con perdida. Esa es la logica detras de elegir buenos casos para `parametrize`: no se trata de agregar muchos numeros al azar, sino de cubrir situaciones realmente diferentes.

Corramos las pruebas para ver los casos por separado.

In [20]:
!python -m pytest tests/test_marketing.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/taller_calidad
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 4 items                                                              

tests/test_marketing.py::test_calcular_roi_varios_casos[1500-1000-0.5] PASSED [ 25%]
tests/test_marketing.py::test_calcular_roi_varios_casos[1000-1000-0.0] PASSED [ 50%]
tests/test_marketing.py::test_calcular_roi_varios_casos[800-1000--0.2] PASSED [ 75%]
tests/test_marketing.py::test_calcular_roi_inversion_cero PASSED         [100%]

============================== 4 passed in 0.06s ===============================


## Parte 3: Mocks - probando codigo que depende de algo externo

### El problema

Agreguemos una funcion nueva a `src/marketing.py`: una que convierte un monto de dolares a moneda local, usando una tasa de cambio. En un proyecto real, esa tasa vendria de una llamada a una API externa. Para simularlo sin depender de internet, la funcion va a devolver un valor aleatorio cada vez que se llama, tal como lo haria una tasa de cambio real, que cambia todo el tiempo.

In [21]:
%%writefile src/marketing.py
"""Modulo de metricas de marketing."""

import random


def calcular_roi(ingresos, inversion):
    """
    Calcula el retorno de inversion (ROI) de una campana.

    Parameters
    ----------
    ingresos : float
        Ingresos generados por la campana.
    inversion : float
        Monto invertido en la campana.

    Returns
    -------
    float
        El ROI, calculado como (ingresos - inversion) / inversion.

    Raises
    ------
    ValueError
        Si la inversion es igual a cero.
    """
    if inversion == 0:
        raise ValueError("La inversion no puede ser cero")
    return round((ingresos - inversion) / inversion, 4)


def obtener_tasa_cambio_usd_a_local():
    """
    Obtiene la tasa de cambio actual de USD a moneda local.

    En un proyecto real, esta funcion haria una llamada a una API externa,
    algo como requests.get("https://api.exchange.com/usd"). Aqui la
    simulamos con un valor aleatorio, pero la idea es la misma: el
    resultado depende de un servicio fuera de nuestro control, y puede
    cambiar en cualquier momento.
    """
    return round(random.uniform(3.5, 4.0), 2)


def convertir_a_moneda_local(monto_usd):
    """
    Convierte un monto en dolares a moneda local.

    Parameters
    ----------
    monto_usd : float
        Monto en dolares.

    Returns
    -------
    float
        Monto equivalente en moneda local, segun la tasa de cambio actual.
    """
    tasa = obtener_tasa_cambio_usd_a_local()
    return round(monto_usd * tasa, 2)

Overwriting src/marketing.py


Probemos `convertir_a_moneda_local` un par de veces seguidas.

In [22]:
from src.marketing import convertir_a_moneda_local

print(convertir_a_moneda_local(100))
print(convertir_a_moneda_local(100))

396.0
397.0


Los dos resultados son distintos, aunque llamamos a la funcion con el mismo monto. Ahora pensemos como escribiriamos una prueba para esto. Si escribimos algo como:

```python
def test_convertir_a_moneda_local():
    assert convertir_a_moneda_local(100) == 375.0
```

esta prueba va a fallar la mayoria de las veces, no porque la funcion este mal, sino porque depende de un valor que cambia cada vez que se ejecuta. Esto es exactamente el tipo de situacion que ya habiamos anticipado en la sesion 3, en la lista de errores comunes: **no se debe probar codigo que depende de un servicio externo real**, porque el resultado deja de ser predecible.

### La idea de un mock

Un mock es como un doble de riesgo en una pelicula: en una escena peligrosa, no se usa al actor real, se usa a alguien que lo reemplaza temporalmente y que se comporta de forma controlada. Un mock hace lo mismo con una funcion: durante la prueba, reemplaza temporalmente una funcion real (como `obtener_tasa_cambio_usd_a_local`) por una version que nosotros controlamos, y que siempre devuelve el mismo valor.

Asi, la prueba deja de depender del azar (o de una API real), y pasa a probar unicamente la logica que nos interesa: que `convertir_a_moneda_local` multiplique correctamente el monto por la tasa.

Agreguemos esta prueba a `tests/test_marketing.py`, usando `unittest.mock.patch` para reemplazar `obtener_tasa_cambio_usd_a_local` durante la prueba.

In [23]:
# Creamos el archivo de pruebas para el módulo marketing
%%writefile tests/test_marketing.py
"""Pruebas unitarias para src/marketing.py"""

from unittest.mock import patch  # Importamos patch para simular funciones externas durante las pruebas

import pytest  # Importamos pytest para crear y ejecutar pruebas

from src.marketing import calcular_roi, convertir_a_moneda_local  # Importamos las funciones que queremos probar


@pytest.mark.parametrize(  # Indicamos que la prueba se ejecutará con varios conjuntos de datos
    "ingresos, inversion, roi_esperado",  # Definimos los parámetros que recibirá la prueba
    [
        (1500, 1000, 0.5),  # Caso 1: ROI del 50%
        (1000, 1000, 0.0),  # Caso 2: no hay ganancia ni pérdida
        (800, 1000, -0.2),  # Caso 3: ROI negativo del 20%
    ],
) # justo abajo de @pytest.mark.parametrize
def test_calcular_roi_varios_casos(ingresos, inversion, roi_esperado):  # Probamos el cálculo del ROI con diferentes casos
    assert calcular_roi(ingresos, inversion) == roi_esperado  # Comprobamos que el ROI calculado sea el esperado


def test_calcular_roi_inversion_cero():  # Probamos el comportamiento cuando la inversión es cero
    with pytest.raises(ValueError):  # Esperamos que la función genere un ValueError
        calcular_roi(1500, 0)  # Intentamos calcular el ROI con una inversión inválida


def test_convertir_a_moneda_local():  # Probamos la conversión de dólares a moneda local
    with patch("src.marketing.obtener_tasa_cambio_usd_a_local", return_value=3.75):  # Simulamos la tasa de cambio sin consultar una fuente externa
        resultado = convertir_a_moneda_local(100)  # Convertimos 100 USD usando la tasa simulada
        assert resultado == 375.0  # Comprobamos que el resultado de la conversión sea el esperado

Overwriting tests/test_marketing.py


Un detalle sobre `patch`: el texto `"src.marketing.obtener_tasa_cambio_usd_a_local"` le dice a pytest exactamente que funcion reemplazar, indicando en que modulo vive. Mientras el bloque `with patch(...)` esta activo, cualquier llamada a esa funcion (incluso desde adentro de `convertir_a_moneda_local`) devuelve `3.75` en vez de un valor aleatorio. Al salir del bloque `with`, todo vuelve a la normalidad.

Corramos las pruebas varias veces seguidas, para confirmar que ahora el resultado es siempre el mismo, sin importar cuantas veces se ejecute.

In [24]:
!python -m pytest tests/test_marketing.py -v
!python -m pytest tests/test_marketing.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/taller_calidad
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 5 items                                                              

tests/test_marketing.py::test_calcular_roi_varios_casos[1500-1000-0.5] PASSED [ 20%]
tests/test_marketing.py::test_calcular_roi_varios_casos[1000-1000-0.0] PASSED [ 40%]
tests/test_marketing.py::test_calcular_roi_varios_casos[800-1000--0.2] PASSED [ 60%]
tests/test_marketing.py::test_calcular_roi_inversion_cero PASSED         [ 80%]
tests/test_marketing.py::test_convertir_a_moneda_local PASSED            [100%]

============================== 5 passed in 0.07s ===============================
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, plu

## Parte 4 : cuanto del codigo esta realmente probado

Una pregunta razonable en este punto es: de todo el codigo que escribimos, cuanto esta efectivamente cubierto por pruebas? La libreria `pytest-cov` responde exactamente eso.

In [25]:
!pip install pytest-cov -q # install pytest-cov

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.5/257.5 kB 13.5 MB/s eta 0:00:00


In [26]:
!python -m pytest --cov=src tests/

# --cov=src → Activa coverage y le indica que mida la cobertura del código que está dentro de src/.
# tests/ → Le indica a pytest dónde buscar y ejecutar las pruebas.

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/drive/MyDrive/taller_calidad
plugins: cov-7.1.0, typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 7 items                                                              

tests/test_marketing.py .....                                            [ 71%]
tests/test_procesamiento.py ..                                           [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.13.15-final-0 _______________

Name                   Stmts   Miss  Cover
------------------------------------------
src/marketing.py          10      1    90%
src/procesamiento.py      10      0   100%
------------------------------------------
TOTAL                     20      1    95%
============================== 7 passed in 0.64s ===============================

La columna `Cover` muestra el porcentaje de lineas de cada archivo que se ejecutaron durante las pruebas. Un numero alto no garantiza que las pruebas sean buenas (se puede ejecutar una linea sin verificar realmente su resultado), pero un numero muy bajo si es una señal clara de que hay codigo sin ninguna prueba encima.

## Errores comunes

- **Mockear de mas**: si mockeas la funcion que en realidad queres probar, la prueba deja de verificar algo real. Solo se mockea lo externo (una API, una base de datos, un servicio de terceros), nunca la logica propia que es el motivo de la prueba.
- **Fixtures donde no hacen falta**: si una prueba necesita datos unicos y muy especificos, forzarla a usar un fixture compartido solo complica las cosas. Los fixtures ayudan cuando la preparacion se repite, no como regla general.
- **Parametrize con casos que no suman**: agregar diez variantes casi identicas no aporta mas confianza que tener tres. Conviene elegir casos que representen situaciones realmente distintas: un caso tipico, un caso limite, un caso de error.

## Guardando el avance en Git

In [ ]:
!git status

In [ ]:
!git add .
!git commit -m "Agrega fixtures, tests parametrizados y mocks a la suite de pruebas"

In [ ]:
!git log --oneline

## Cierre de la sesion

Repasamos hoy tres herramientas de pytest, cada una resolviendo un problema concreto:

- **Fixtures**: evitan repetir la preparacion de datos en cada prueba, centralizandola en `conftest.py`.
- **Parametrize**: permiten probar varios casos con una sola funcion de test, en vez de copiar y pegar.
- **Mocks**: permiten probar codigo que depende de algo externo (una API, un valor aleatorio, una base de datos), sin depender de verdad de ese servicio durante la prueba.

Como bonus, vimos como medir la cobertura de pruebas del proyecto con `pytest-cov`.

Con esto, `taller_calidad` queda con una suite de pruebas bastante mas solida que la que teniamos al cerrar la sesion 3: no solo mas pruebas, sino pruebas mas faciles de mantener y de extender.

### Antes de terminar, verifica que puedes responder:

- Que problema resuelve un fixture, y donde conviene definir uno que se comparte entre varios archivos de prueba.
- Por que `parametrize` es preferible a escribir una funcion de test por cada caso.
- Por que no se debe probar una funcion que depende de una API real sin usar un mock.